# 8. Click Bias 与 IPS：怎样从位置偏置点击估计反事实相关性？

## 面试回答主线

点击不是纯相关性标签，因为排名靠前的文档更容易被看见。若 position 1 的 examination propensity 是 0.9、position 3 只有 0.25，直接 CTR 会高估长期占据首位的普通结果。IPS 为每次点击乘 `1 / propensity`，把低曝光位置中发生的点击还原为更强证据；未点击贡献为零。面试时我会在六篇真实搜索结果的 impression 日志上输出 position、propensity、权重和逐文档估计，并与 CTR 基线比较。极小 propensity 会制造高方差，应做 positivity 检查、权重 clipping 或 self-normalized/DR 估计。生产上 propensity 必须来自真实随机化或可靠 logging policy，而不能事后猜测。

## 1. 真实案例：六篇文档、二十四次曝光与位置点击

D1～D3 是真正能解决问题的结果，但多出现在 position 2/3；D4～D6 标题吸睛却内容普通，长期占据 position 1。每篇文档有四次 impression，position propensity 来自一次小流量随机化实验。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示点击日志和反事实权重
documents = {"D1": {"title": "退款原路返回的完整步骤", "relevant": 1}, "D2": {"title": "慢查询执行计划排查", "relevant": 1}, "D3": {"title": "RAG 答案如何附来源", "relevant": 1}, "D4": {"title": "退款常见问题大全", "relevant": 0}, "D5": {"title": "数据库热门技巧", "relevant": 0}, "D6": {"title": "大模型最新资讯", "relevant": 0}}  # 定义六篇有人工相关性标签的真实文档
patterns = {"D1": [(2, 1), (3, 0), (2, 1), (3, 0)], "D2": [(3, 1), (2, 0), (3, 1), (2, 0)], "D3": [(2, 1), (2, 1), (3, 0), (3, 0)], "D4": [(1, 1), (1, 1), (1, 1), (1, 0)], "D5": [(1, 1), (1, 1), (1, 1), (1, 0)], "D6": [(1, 1), (1, 1), (1, 0), (1, 0)]}  # 定义每篇文档四次曝光的位置与点击
propensity_by_position = {1: 0.90, 2: 0.45, 3: 0.25}  # 记录用户真正查看各位置的实验估计概率
impressions = []  # 初始化逐次曝光日志
for doc_id, observations in patterns.items():  # 遍历六篇文档的曝光序列
    for index, (position, click) in enumerate(observations, start=1):  # 展开每次位置与点击结果
        impressions.append({"impression_id": f"{doc_id}-I{index}", "doc_id": doc_id, "position": position, "click": click, "propensity": propensity_by_position[position]})  # 保存可用于 IPS 的完整 logging policy 字段
preview = [{"曝光": row["impression_id"], "文档": documents[row["doc_id"]]["title"], "position": row["position"], "click": row["click"], "propensity": row["propensity"]} for row in impressions]  # 汇总真实曝光的业务字段
print("点击偏置日志前十二条：")  # 输出真实案例标题
pprint(preview[:12], sort_dicts=False)  # 展示相关文档在低曝光位置仍产生点击

点击偏置日志前十二条：
[{'曝光': 'D1-I1',
  '文档': '退款原路返回的完整步骤',
  'position': 2,
  'click': 1,
  'propensity': 0.45},
 {'曝光': 'D1-I2',
  '文档': '退款原路返回的完整步骤',
  'position': 3,
  'click': 0,
  'propensity': 0.25},
 {'曝光': 'D1-I3',
  '文档': '退款原路返回的完整步骤',
  'position': 2,
  'click': 1,
  'propensity': 0.45},
 {'曝光': 'D1-I4',
  '文档': '退款原路返回的完整步骤',
  'position': 3,
  'click': 0,
  'propensity': 0.25},
 {'曝光': 'D2-I1',
  '文档': '慢查询执行计划排查',
  'position': 3,
  'click': 1,
  'propensity': 0.25},
 {'曝光': 'D2-I2',
  '文档': '慢查询执行计划排查',
  'position': 2,
  'click': 0,
  'propensity': 0.45},
 {'曝光': 'D2-I3',
  '文档': '慢查询执行计划排查',
  'position': 3,
  'click': 1,
  'propensity': 0.25},
 {'曝光': 'D2-I4',
  '文档': '慢查询执行计划排查',
  'position': 2,
  'click': 0,
  'propensity': 0.45},
 {'曝光': 'D3-I1',
  '文档': 'RAG 答案如何附来源',
  'position': 2,
  'click': 1,
  'propensity': 0.45},
 {'曝光': 'D3-I2',
  '文档': 'RAG 答案如何附来源',
  'position': 2,
  'click': 1,
  'propensity': 0.45},
 {'曝光': 'D3-I3',
  '文档': 'RAG 答案如何附来源',
  'position': 3,

## 2. Baseline（基线）：直接按 observed CTR 排序

CTR 把“被看见”和“看见后愿意点击”混在一起。D4、D5 在 position 1 获得 75% CTR，因此压过低位相关结果；下面逐文档统计点击、曝光和平均位置。

In [2]:
baseline_rows = []  # 收集六篇文档的 observed CTR 统计
for doc_id, metadata in documents.items():  # 遍历全部候选文档
    rows = [row for row in impressions if row["doc_id"] == doc_id]  # 取得当前文档四次曝光记录
    clicks = sum(row["click"] for row in rows)  # 统计观察到的点击次数
    ctr = clicks / len(rows)  # 直接用点击除曝光得到偏置 CTR
    average_position = sum(row["position"] for row in rows) / len(rows)  # 计算文档长期获得的平均展示位置
    baseline_rows.append({"doc": doc_id, "标题": metadata["title"], "人工相关": metadata["relevant"], "点击": clicks, "曝光": len(rows), "CTR": ctr, "平均position": average_position})  # 保存逐文档偏置基线
baseline_ranking = sorted(baseline_rows, key=lambda row: (-row["CTR"], row["doc"]))  # 按 observed CTR 生成错误反事实排序
print("Observed CTR 基线排序：")  # 标注当前输出属于点击偏置基线
pprint(baseline_ranking, sort_dicts=False)  # 展示首位普通文档如何压过低位相关文档

Observed CTR 基线排序：
[{'doc': 'D4',
  '标题': '退款常见问题大全',
  '人工相关': 0,
  '点击': 3,
  '曝光': 4,
  'CTR': 0.75,
  '平均position': 1.0},
 {'doc': 'D5',
  '标题': '数据库热门技巧',
  '人工相关': 0,
  '点击': 3,
  '曝光': 4,
  'CTR': 0.75,
  '平均position': 1.0},
 {'doc': 'D1',
  '标题': '退款原路返回的完整步骤',
  '人工相关': 1,
  '点击': 2,
  '曝光': 4,
  'CTR': 0.5,
  '平均position': 2.5},
 {'doc': 'D2',
  '标题': '慢查询执行计划排查',
  '人工相关': 1,
  '点击': 2,
  '曝光': 4,
  'CTR': 0.5,
  '平均position': 2.5},
 {'doc': 'D3',
  '标题': 'RAG 答案如何附来源',
  '人工相关': 1,
  '点击': 2,
  '曝光': 4,
  'CTR': 0.5,
  '平均position': 2.5},
 {'doc': 'D6',
  '标题': '大模型最新资讯',
  '人工相关': 0,
  '点击': 2,
  '曝光': 4,
  'CTR': 0.5,
  '平均position': 1.0}]


## 3. 手写 IPS：输出每次曝光的 propensity、权重与贡献

IPS contribution 为 `click / propensity`。低位点击得到更大权重，但低位未点击仍为零；逐文档再除固定 impression 数得到 Horvitz-Thompson relevance proxy。

In [3]:
ips_ledger = []  # 收集二十四次曝光的反事实权重中间量
for row in impressions:  # 遍历原始点击日志
    inverse_weight = 1.0 / row["propensity"]  # 根据 logging policy 计算逆曝光概率
    contribution = row["click"] * inverse_weight  # 只有实际点击事件产生反事实奖励贡献
    ips_ledger.append({**row, "inverse_weight": inverse_weight, "ips_contribution": contribution})  # 保存位置、propensity、权重和贡献
print("D1 与 D4 的 IPS 曝光账本：")  # 输出核心估计中间量标题
pprint([{**row, "inverse_weight": round(row["inverse_weight"], 3), "ips_contribution": round(row["ips_contribution"], 3)} for row in ips_ledger if row["doc_id"] in {"D1", "D4"}], sort_dicts=False)  # 对比低位相关点击与首位普通点击的权重

D1 与 D4 的 IPS 曝光账本：
[{'impression_id': 'D1-I1',
  'doc_id': 'D1',
  'position': 2,
  'click': 1,
  'propensity': 0.45,
  'inverse_weight': 2.222,
  'ips_contribution': 2.222},
 {'impression_id': 'D1-I2',
  'doc_id': 'D1',
  'position': 3,
  'click': 0,
  'propensity': 0.25,
  'inverse_weight': 4.0,
  'ips_contribution': 0.0},
 {'impression_id': 'D1-I3',
  'doc_id': 'D1',
  'position': 2,
  'click': 1,
  'propensity': 0.45,
  'inverse_weight': 2.222,
  'ips_contribution': 2.222},
 {'impression_id': 'D1-I4',
  'doc_id': 'D1',
  'position': 3,
  'click': 0,
  'propensity': 0.25,
  'inverse_weight': 4.0,
  'ips_contribution': 0.0},
 {'impression_id': 'D4-I1',
  'doc_id': 'D4',
  'position': 1,
  'click': 1,
  'propensity': 0.9,
  'inverse_weight': 1.111,
  'ips_contribution': 1.111},
 {'impression_id': 'D4-I2',
  'doc_id': 'D4',
  'position': 1,
  'click': 1,
  'propensity': 0.9,
  'inverse_weight': 1.111,
  'ips_contribution': 1.111},
 {'impression_id': 'D4-I3',
  'doc_id': 'D4',
  'posit

## 4. 逐文档 IPS 估计与反事实排序

每篇文档曝光次数相同，因此用 contribution 平均值直接比较。结果表同时保留 CTR、IPS、平均 propensity 与人工相关性，避免只展示最终 top-k。

In [4]:
ips_rows = []  # 收集六篇文档的反事实相关性估计
for baseline in baseline_rows:  # 复用同一文档集合与 observed CTR
    rows = [row for row in ips_ledger if row["doc_id"] == baseline["doc"]]  # 取得当前文档的全部加权曝光
    ips_score = sum(row["ips_contribution"] for row in rows) / len(rows)  # 计算 Horvitz-Thompson 平均点击奖励
    average_propensity = sum(row["propensity"] for row in rows) / len(rows)  # 计算该文档实际获得的平均查看概率
    ips_rows.append({"doc": baseline["doc"], "标题": baseline["标题"], "人工相关": baseline["人工相关"], "CTR": baseline["CTR"], "平均propensity": average_propensity, "IPS分数": ips_score})  # 保存逐文档校正前后指标
ips_ranking = sorted(ips_rows, key=lambda row: (-row["IPS分数"], row["doc"]))  # 按反事实奖励生成新排序
print("IPS 反事实排序：")  # 输出核心方案结果标题
pprint([{**row, "IPS分数": round(row["IPS分数"], 3)} for row in ips_ranking], sort_dicts=False)  # 展示低位相关文档获得曝光校正后的排名

IPS 反事实排序：
[{'doc': 'D2',
  '标题': '慢查询执行计划排查',
  '人工相关': 1,
  'CTR': 0.5,
  '平均propensity': 0.35,
  'IPS分数': 2.0},
 {'doc': 'D1',
  '标题': '退款原路返回的完整步骤',
  '人工相关': 1,
  'CTR': 0.5,
  '平均propensity': 0.35,
  'IPS分数': 1.111},
 {'doc': 'D3',
  '标题': 'RAG 答案如何附来源',
  '人工相关': 1,
  'CTR': 0.5,
  '平均propensity': 0.35,
  'IPS分数': 1.111},
 {'doc': 'D4',
  '标题': '退款常见问题大全',
  '人工相关': 0,
  'CTR': 0.75,
  '平均propensity': 0.9,
  'IPS分数': 0.833},
 {'doc': 'D5',
  '标题': '数据库热门技巧',
  '人工相关': 0,
  'CTR': 0.75,
  '平均propensity': 0.9,
  'IPS分数': 0.833},
 {'doc': 'D6',
  '标题': '大模型最新资讯',
  '人工相关': 0,
  'CTR': 0.5,
  '平均propensity': 0.9,
  'IPS分数': 0.556}]


## 5. 结果解读：用人工相关 top-3 检查是否真正去偏

CTR top-3 只包含一篇真正相关文档，IPS top-3 恢复 D1～D3。这个受控结果依赖正确 propensity 和充分 support；IPS 不是把所有低位点击都当成真相关，而是校正观察机会。

In [5]:
baseline_top3 = [row["doc"] for row in baseline_ranking[:3]]  # 取得 observed CTR 的前三篇文档
ips_top3 = [row["doc"] for row in ips_ranking[:3]]  # 取得反事实 IPS 的前三篇文档
relevant_docs = {doc_id for doc_id, metadata in documents.items() if metadata["relevant"] == 1}  # 建立人工标注的真正相关集合
baseline_relevant_at3 = len(set(baseline_top3) & relevant_docs)  # 计算 CTR 排序前三的人工相关数量
ips_relevant_at3 = len(set(ips_top3) & relevant_docs)  # 计算 IPS 排序前三的人工相关数量
comparison = [{"doc": row["doc"], "标题": row["标题"], "人工相关": row["人工相关"], "CTR排名": next(index + 1 for index, item in enumerate(baseline_ranking) if item["doc"] == row["doc"]), "IPS排名": next(index + 1 for index, item in enumerate(ips_ranking) if item["doc"] == row["doc"]), "IPS分数": round(row["IPS分数"], 3)} for row in ips_rows]  # 构造逐文档排名变化表
print("CTR 与 IPS 逐文档对照：")  # 输出结果解读标题
pprint(comparison, sort_dicts=False)  # 展示位置偏置校正如何改变每篇文档
print({"CTR相关@3": baseline_relevant_at3, "IPS相关@3": ips_relevant_at3, "目标相关集合": sorted(relevant_docs)})  # 汇总 top-k 质量差异

CTR 与 IPS 逐文档对照：
[{'doc': 'D1',
  '标题': '退款原路返回的完整步骤',
  '人工相关': 1,
  'CTR排名': 3,
  'IPS排名': 2,
  'IPS分数': 1.111},
 {'doc': 'D2',
  '标题': '慢查询执行计划排查',
  '人工相关': 1,
  'CTR排名': 4,
  'IPS排名': 1,
  'IPS分数': 2.0},
 {'doc': 'D3',
  '标题': 'RAG 答案如何附来源',
  '人工相关': 1,
  'CTR排名': 5,
  'IPS排名': 3,
  'IPS分数': 1.111},
 {'doc': 'D4',
  '标题': '退款常见问题大全',
  '人工相关': 0,
  'CTR排名': 1,
  'IPS排名': 4,
  'IPS分数': 0.833},
 {'doc': 'D5',
  '标题': '数据库热门技巧',
  '人工相关': 0,
  'CTR排名': 2,
  'IPS排名': 5,
  'IPS分数': 0.833},
 {'doc': 'D6',
  '标题': '大模型最新资讯',
  '人工相关': 0,
  'CTR排名': 6,
  'IPS排名': 6,
  'IPS分数': 0.556}]
{'CTR相关@3': 1, 'IPS相关@3': 3, '目标相关集合': ['D1', 'D2', 'D3']}


## 6. 失败案例与修正：propensity 接近零导致单次权重爆炸

一个 position 10 的点击若 propensity=0.01，会产生 100 倍贡献，单条日志即可支配排序。修正先做 positivity 门禁，再将 propensity 下限裁到 0.1；更稳健的生产方案还包括 self-normalized IPS 和 doubly robust estimator。

In [6]:
rare_impression = {"impression_id": "RARE-1", "doc_id": "D6", "position": 10, "click": 1, "propensity": 0.01}  # 构造几乎从不曝光但偶然点击的日志
unclipped_weight = 1.0 / rare_impression["propensity"]  # 复现极小 propensity 造成的高方差逆权重
minimum_propensity = 0.10  # 设置离线估计允许的最小有效曝光概率
clipped_propensity = max(rare_impression["propensity"], minimum_propensity)  # 对低于 support 门槛的概率执行裁剪
clipped_weight = 1.0 / clipped_propensity  # 计算裁剪后的有限逆权重
support_warning = rare_impression["propensity"] < minimum_propensity  # 标记该日志缺乏可靠反事实支持
print({"失败_propensity": rare_impression["propensity"], "未裁剪权重": unclipped_weight, "支持不足": support_warning, "修正_propensity": clipped_propensity, "裁剪后权重": clipped_weight})  # 展示高方差失败与权重裁剪

{'失败_propensity': 0.01, '未裁剪权重': 100.0, '支持不足': True, '修正_propensity': 0.1, '裁剪后权重': 10.0}


## 7. 生产差距与最小回归检查

真实 propensity 应来自随机交换、interleaving 或已知 logging policy，并与每次 impression 一起落盘。若某文档在某位置从未出现，就不满足 positivity，IPS 无法凭空推断；权重裁剪会引入偏差，需要报告 bias-variance 权衡。下面的断言只验证本实验的日志规模、位置权重、排序恢复和极小概率失败。

In [7]:
assert len(documents) >= 6 and len(impressions) == 24  # 确认真实文档与曝光日志数量满足教学要求
assert all(row["ips_contribution"] >= 0.0 for row in ips_ledger)  # 确认逐曝光 IPS 贡献由点击和正 propensity 合法生成
assert baseline_relevant_at3 < ips_relevant_at3  # 确认位置偏置校正改善人工相关 top-3
assert set(ips_top3) == relevant_docs  # 确认受控数据中 IPS 恢复三篇真正相关文档
assert unclipped_weight == 100.0 and clipped_weight == 10.0  # 确认极小 propensity 权重爆炸和裁剪结果
assert support_warning is True  # 确认缺乏反事实 support 的日志被显式标记
print("回归检查通过：position propensity、逐曝光 IPS、反事实排序与权重裁剪均已验证。")  # 输出最终验收结论

回归检查通过：position propensity、逐曝光 IPS、反事实排序与权重裁剪均已验证。
